# Part 4: Which request goes first? (shortest prompt first, with aging)

Fill the Part 4 TODOs in `vidur/scheduler/replica_scheduler/spf_replica_scheduler.py` first (`python tests/test_spf.py`). Then run SPF at five aging values, from the repo root:

```bash
for A in 0 25 50 100 200; do
  SCHED=spf CHUNK=512 AGING=$A QPS=5.53 bash scripts/run_default.sh traces/azure_conversation.csv
done
```

Runs go one after another, about 3 minutes each. If you have plenty of RAM, you can run them in the background instead (add `&` after each command, then `wait`), but each run needs a few GB: four at once ran out of memory on a 16 GB laptop.

Compare them with your Part 3 Sarathi run at C = 512, which starts requests in arrival order.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if not (ROOT / "traces").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from analysis.replay import describe_run, load_run
from analysis.workload import load_trace

SURGE = (1350, 2230)   # seconds of trace time where one GPU falls behind (Parts 1-3)

trace = load_trace(ROOT / "traces" / "azure_conversation.csv")

# Your Part 3 Sarathi C=512 run and your five SPF runs, any order.
RUN_DIRS = [ROOT / "simulator_output" / d for d in [
    "REPLACE_SARATHI_C512",
    "REPLACE_AGING0", "REPLACE_AGING25", "REPLACE_AGING50", "REPLACE_AGING100", "REPLACE_AGING200",
]]
def _order(name):  # arrival order first, then by aging
    return float(name.split("aging=")[1]) if "aging=" in name else -1.0

runs = {describe_run(d): load_run(d, trace)["requests"] for d in RUN_DIRS}
runs = dict(sorted(runs.items(), key=lambda kv: _order(kv[0])))
print(list(runs))

## Median vs worst case

In [ ]:
print(f"{'run':26s}{'surge median':>14s}{'P99':>9s}{'worst':>10s}{'over 60 s':>11s}")
for name, req in runs.items():
    surge = req[(req.arrived_at >= SURGE[0]) & (req.arrived_at <= SURGE[1])]
    print(f"{name:26s}{surge.wait.median():12.1f} s{req.wait.quantile(.99):7.1f} s"
          f"{req.wait.max():8.1f} s{int((req.wait > 60).sum()):11d}")

fig, ax = plt.subplots(figsize=(10, 3.5))
for name, req in runs.items():
    xs = np.sort(req.wait.to_numpy())
    ax.plot(xs, 1 - np.arange(1, len(xs) + 1) / len(xs), label=name)
ax.set_xscale("symlog", linthresh=1)
ax.set_yscale("log")
ax.set_xlabel("wait (s)")
ax.set_ylabel("fraction of requests waiting longer")
ax.legend(frameon=False)
plt.show()

## Who waits longest under shortest-prompt-first?

In [ ]:
spf = next(req for name, req in runs.items() if name.endswith("aging=0"))
print(spf.sort_values("wait", ascending=False)
      .head(8)[["arrived_at", "num_prefill_tokens", "num_decode_tokens", "wait"]]
      .round(1).to_string(index=False))

## Questions

**Q1.** Fill in the table. The surge is the requests that arrive between t = 1350 s and t = 2230 s. Which order gives the lowest surge median wait, and which gives the lowest worst wait? Does any order meet the 2 s P99 target?

| Order | Surge median wait | P99 wait | Worst wait | Requests over 60 s |
|---|---|---|---|---|
| Arrival order (Part 3) | | | | |
| Aging 0 (plain shortest-first) | | | | |
| Aging 25 | | | | |
| Aging 50 | | | | |
| Aging 100 | | | | |
| Aging 200 | | | | |

**Your answer:**

**Q2.** Under shortest-prompt-first (aging 0), how long are the prompts of the requests that wait longest, and when did they arrive? In one sentence, why do they wait so long? P99 summarizes the slowest 1% of requests, 194 of 19,366: compare aging 0's count of requests over 60 s with 194 to explain why its P99 still looks better than arrival order's.

**Your answer:**

**Q3.** From your table: which is the smallest aging that keeps the surge median wait under 1 s and has at most 5 requests waiting over 60 s? What does it cost against arrival order in P99 and worst wait? Aging 25 has a higher P99 than plain SPF: use the two counts of requests over 60 s and the 194 from Q2 to explain why.

**Your answer:**